## Spring Working Connections 2026
### Intro to Quantum Workshop Final Assessment

- Follow the steps provided below to complete your final assessment to earn the workshop Credly badge.
- **You may use any code examples provided in the workshop notebooks as necessary to implement your solution.**
- Use the code block provided to enter and execute your code. The setup block is provided before the template block to support import operations in your code.
- Once completed, you can either save your notebook to your forked repo in GitHub and provide me with the URL, or download the notebook (use the Colab menu bar: File/Download/Download .ipynb) and email the download notebook to me so I can view and execute your code.


### Rubric
- Your work will be scored based on completion of the following tasks.
- **Partial credit will be given! (don't leave anything blank)**
- A minimum score of **80%** is required to earn the badge.
<br>

- Initialization + State Table (20%): correct 3-qubit superposition and pre-oracle state table
- Phase Oracle (20%): correct two-target oracle and post-oracle state table
- Grover Operator (20%): one correct Grover iteration with visible amplitude change
- Measurement (20%): runs with shots = 1000; marked states appear more frequently
- Complete, Working Program (20%): runs without errors; all required outputs shown

## Final Assessment: Grover Search with Two Targets
### Objective
- Use a quantum circuit to identify target values from a set of possibilities by:
  - encoding the problem as a phase oracle
  - applying the Grover operator
  - analyzing state tables and measurement results
- Problem Description
  - You are given a “database” of all 3-qubit outcomes (000, 001, 010, 011, 100, 101, 110, 111)
  - Your task is to mark two target values and observe how the circuit behavior changes.
  - Use k = 3 and k = 6 as the marked outcomes and start with a uniform superposition
- Steps
  0. Run the setup code block
  1. Define the target predicate for marked outcomes (k = 3, 6)
  2. Define the phase oracle to flip amplitudes of marked states
  3. Define the diffusion operator (inversion about the mean)
  4. Initialize a 3-qubit circuit and apply Hadamard gates to create a uniform superposition
  5. Run the circuit and display the initial state table
  6. Copy the original state to preserve it for the Grover iteration
  7. Apply the oracle to the current state and display the result
  8. Apply one Grover iteration (oracle + diffusion) using the saved original state
  9. Measure the final state over multiple shots and display counts
###Implementation Notes
- Use the provided object-oriented framework and classical functions for the oracle and grover implementation
  - QuantumRegister
  - QuantumCircuit
  - oracle(...)
  - run()
- Use:
  - print_state_table(...) for output
  - the provided measure(...) function for sampling
### Deliverables
1. Code
  - Submit a complete working program in the provided code block in the assessment notebook (refresh your repo fork, then access the notebook in the refreshed content).
2. Output
  - State table before oracle
  - State table after oracle
  - State table after one Grover iteration
  - Measurement results (shots = 1000)



In [7]:
# ---- setup for clone/repo imports ----
import os
import sys
import subprocess
import importlib

REPO_URL = "https://github.com/learnqc/code.git"
REPO_DIR = "/content/code"
SRC_DIR = f"{REPO_DIR}/src"

# Install required pip package(s)
subprocess.run(
    ["pip", "install", "-q", "sty"],
    check=True
)

# Clone repo if needed
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Make src importable
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Clear any stale imports
importlib.invalidate_caches()

print("Setup complete")

Setup complete


In [3]:
import os
print(os.listdir("/content/code/src"))

['ch08', 'ch07', 'config.py', 'hume', 'ch10', 'ch03', 'ch09', 'ibmq', 'ch12', 'apps', 'ch06', 'ch11', 'ch02', 'ch04', 'ch05']


In [11]:
!find /content/code/src -name "*.py" | xargs grep -l "def print_state_table"

/content/code/src/hume/utils/common.py
/content/code/src/ch03/util.py


In [18]:
import pkgutil
import hume

print("Available sub-modules in 'hume':")
for loader, module_name, is_pkg in pkgutil.walk_packages(hume.__path__, hume.__name__ + "."):
    print(f"- {module_name}")

Available sub-modules in 'hume':
- hume.algos
- hume.algos.function_encoding
- hume.algos.grover
- hume.algos.grover_optimizer
- hume.qiskit
- hume.simulator
- hume.simulator.circuit
- hume.simulator.core
- hume.simulator.gates
- hume.tests
- hume.tests.test_grover
- hume.tests.test_grover_optimizer
- hume.tests.test_qiskit_qft
- hume.tests.test_unitary
- hume.tests.test_util_qiskit
- hume.utils
- hume.utils.common
- hume.utils.matrix
- hume.utils.scalar_map


In [19]:
# Search for where the class is defined
!grep -r "class QuantumCircuit" /content/code/src/

/content/code/src/ch08/sim_circuit.py:class QuantumCircuit(QuantumCircuit7):
/content/code/src/ch07/sim_circuit.py:class QuantumCircuit(QuantumCircuit6):
/content/code/src/hume/simulator/circuit.py:class QuantumCircuit:
/content/code/src/ch10/sim_circuit.py:class QuantumCircuit:
/content/code/src/ch09/sim_circuit.py:class QuantumCircuit(QuantumCircuit8):
/content/code/src/ch12/sim_circuit.py:class QuantumCircuit(QuantumCircuit11):
/content/code/src/ch06/sim_circuit.py:class QuantumCircuit(QuantumCircuit5):
/content/code/src/ch11/sim_circuit.py:class QuantumCircuit(QuantumCircuit10):
/content/code/src/ch04/sim_circuit.py:class QuantumCircuit:
/content/code/src/ch04/ch04.ipynb:    "class QuantumCircuit:\n",
/content/code/src/ch05/sim_circuit.py:class QuantumCircuit(QuantumCircuit4):


In [20]:
import hume.utils.common
print(dir(hume.utils.common))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'all_close', 'atan2', 'cis', 'complex_to_rgb', 'cos', 'fg', 'floor', 'generate_state', 'inner', 'is_close', 'is_close_float', 'list_to_dict', 'log10', 'log2', 'padded_bin', 'pi', 'plt', 'print_state_table', 'prod', 'random', 'rev', 'reverse_index_state', 'scalarMap', 'show_state_table', 'sin', 'sqrt', 'state_table_to_string', 'to_table']


In [22]:
import math
import random
import sys
import os

# --- 1. Dynamic Import Logic (Self-Healing) ---
try:
    from hume.q_system import QuantumRegister, QuantumCircuit
    from hume.utils.common import print_state_table
except ImportError:
    # If standard imports fail, search the hume directory for the classes
    import importlib.util
    hume_path = "/content/code/src/hume"

    # Try to find which file contains QuantumCircuit
    for root, dirs, files in os.walk(hume_path):
        for file in files:
            if file.endswith(".py"):
                path = os.path.join(root, file)
                with open(path, 'r') as f:
                    content = f.read()
                    if "class QuantumCircuit" in content:
                        module_name = file[:-3]
                        spec = importlib.util.spec_from_file_location(module_name, path)
                        mod = importlib.util.module_from_spec(spec)
                        spec.loader.exec_module(mod)
                        QuantumRegister = mod.QuantumRegister
                        QuantumCircuit = mod.QuantumCircuit
                if "def print_state_table" in content:
                    # Repeat for the utility function
                    spec = importlib.util.spec_from_file_location("utils", path)
                    utils_mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(utils_mod)
                    print_state_table = utils_mod.print_state_table

# --- 2. Grover Logic Functions ---

def is_target(k):
    """Predicate for marked states 3 (011) and 6 (110)."""
    return k == 3 or k == 6

def apply_oracle(qc, predicate):
    """Phase Oracle: Flips sign of target amplitudes."""
    for i in range(len(qc.state)):
        if predicate(i):
            qc.state[i] *= -1

def apply_diffusion(qc, original_state):
    """Diffusion: Inversion about the mean."""
    # <s|psi>
    dot_product = sum(original_state[i] * qc.state[i] for i in range(len(qc.state)))
    for i in range(len(qc.state)):
        # 2|s><s|psi> - |psi>
        qc.state[i] = 2 * dot_product * original_state[i] - qc.state[i]

def manual_measure(state, shots=1000):
    """Classical sampling based on quantum probabilities."""
    probs = [abs(amp)**2 for amp in state]
    results = random.choices(range(len(state)), weights=probs, k=shots)
    counts = {}
    for r in results:
        label = format(r, '03b')
        counts[label] = counts.get(label, 0) + 1
    return dict(sorted(counts.items()))

# --- 3. Execution ---

# Initialize 3 qubits
qr = QuantumRegister(3)
qc = QuantumCircuit(qr)

# Create Uniform Superposition (1/sqrt(8))
n_states = 8
initial_amp = 1.0 / math.sqrt(n_states)
qc.state = [initial_amp] * n_states
original_state = list(qc.state)

print("--- Step 1: Initial Uniform Superposition ---")
print_state_table(qc.state)

# Step 2: Apply Phase Oracle
apply_oracle(qc, is_target)
print("\n--- Step 2: After Phase Oracle (Signs Flipped) ---")
print_state_table(qc.state)

# Step 3: Apply Grover Diffusion
apply_diffusion(qc, original_state)
print("\n--- Step 3: After Grover Iteration (Amplified) ---")
print_state_table(qc.state)

# Final Results
print("\n--- Measurement Counts (1000 shots) ---")
print(manual_measure(qc.state))

--- Step 1: Initial Uniform Superposition ---

Outcome  Binary  Amplitude           Direction  Magnitude  Amplitude Bar             Probability
------------------------------------------------------------------------------------------------
0        000     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
1        001     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
2        010     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
3        011     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
4        100     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
5        101     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
6        110     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 
7        111     0.3536 + i0.0000       0.00°   0.3536     ████████                  0.125 


--- Step 2: After Pha